# Airline Passenger Forecasting

## Project Overview

Forecasts **daily airline passenger count** (thousands) over a 14-day horizon. Synthetic data spans 2 years with weekly business-travel cycles, summer/holiday peaks, and post-pandemic recovery.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily passenger counts, predict the next 14 days. Airlines need passenger forecasts for capacity planning, crew scheduling, pricing optimization, and fuel procurement.

## Why This Project Matters

Air travel demand directly affects airline revenue and operations. Accurate forecasting enables optimal seat pricing (revenue management), efficient crew rostering, and strategic route planning. Errors cost millions per day across a major airline network.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily passenger counts (thousands)
- Weekly cycle (lower weekends for business travel)
- Strong summer and holiday peaks
- Recovery growth trend
- Random disruption events

| Column | Description |
|--------|-------------|
| `date` | Date |
| `passengers_k` | Daily passenger count (thousands) |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'passengers_k'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

base = 250 + 0.15 * t  # recovery/growth
weekly = np.zeros(N_POINTS)
for i in range(N_POINTS):
    dow = dates[i].dayofweek
    if dow == 0: weekly[i] = 20  # Monday
    elif dow <= 3: weekly[i] = 10  # Tue-Thu
    elif dow == 4: weekly[i] = 25  # Friday peak
    elif dow == 5: weekly[i] = -15  # Saturday low
    else: weekly[i] = 5  # Sunday

# Summer peak + holiday spikes
seasonal = 40 * np.sin(2 * np.pi * (t - 90) / 365)
holiday = np.zeros(N_POINTS)
for i in range(N_POINTS):
    m, d = dates[i].month, dates[i].day
    if (m == 12 and 20 <= d <= 31) or (m == 1 and d <= 3): holiday[i] = 50
    elif m == 11 and 22 <= d <= 28: holiday[i] = 40
    elif m == 3 and 10 <= d <= 20: holiday[i] = 20  # spring break

noise = np.random.normal(0, 12, N_POINTS)
passengers_k = base + weekly + seasonal + holiday + noise
passengers_k = np.maximum(passengers_k, 100).round(1)

df = pd.DataFrame({'date': dates, 'passengers_k': passengers_k})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,passengers_k
0,2022-01-01,251.0
1,2022-01-02,263.5
2,2022-01-03,288.1
3,2022-01-04,238.8
4,2022-01-05,218.0


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('passengers_k Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("passengers_k Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

passengers_k Statistics:
count    730.000000
mean     317.178082
std       45.199321
min      177.800000
25%      288.625000
50%      319.700000
75%      349.525000
max      417.000000
Name: passengers_k, dtype: float64

CV: 0.143
Skewness: -0.314


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:       32.6 | RMSE:       38.3 | MAPE:  8.54%
Seasonal Naive (7)             | MAE:       25.6 | RMSE:       33.4 | MAPE:  6.85%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                            Adjusted R-Squared   R-Squared        RMSE  Time Taken
Model                                                                             
KernelRidge                        5265.034991 -403.925769  321.775444    0.067110
GaussianProcessRegressor           4532.347919 -347.565225  298.543224    0.077787
MLPRegressor                        947.925005  -71.840385  136.474436    0.795586
DecisionTreeRegressor                63.345360   -3.795797   35.018342    0.028100
PassiveAggressiveRegressor           31.198976   -1.322998   24.371913    0.025605
ExtraTreeRegressor                   26.566748   -0.966673   22.424938    0.019279
LinearSVR                            24.214824   -0.785756   21.368607    0.037625
AdaBoostRegressor                    23.697795   -0.745984   21.129312    0.214321
RandomForestRegressor                19.514458   -0.424189   19.083104    0.781439
ExtraTreesRegressor                  18.914311   -0.378024   18.77

## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: catboost
FLAML (catboost)               | MAE:       13.6 | RMSE:       16.7 | MAPE:  4.23%
FLAML Test (catboost)          | MAE:       24.7 | RMSE:       28.6 | MAPE:  6.50%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:       36.0 | RMSE:       41.1 | MAPE:  9.49%
SF AutoETS                     | MAE:       37.2 | RMSE:       40.8 | MAPE:  9.89%
SF SeasonalNaive               | MAE:       38.6 | RMSE:       41.7 | MAPE: 10.33%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
                model       MAE      RMSE      MAPE
     FLAML (catboost) 13.606890 16.705628  4.234004
FLAML Test (catboost) 24.694621 28.638039  6.499580
   Seasonal Naive (7) 25.607143 33.375066  6.850700
   Naive (Last Value) 32.557143 38.296773  8.539265
         SF AutoARIMA 36.016040 41.141382  9.489134
           SF AutoETS 37.206819 40.828544  9.890827
     SF SeasonalNaive 38.614286 41.734809 10.326480

Top 3:
                model       MAE      RMSE     MAPE
     FLAML (catboost) 13.606890 16.705628 4.234004
FLAML Test (catboost) 24.694621 28.638039 6.499580
   Seasonal Naive (7) 25.607143 33.375066 6.850700


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: 23.47, Std: 16.41


## Interpretation and Business Insight

- Airline passengers show clear weekly cycles driven by business travel
- Friday is the peak travel day; Saturday is the lowest
- Summer and Christmas/Thanksgiving create major seasonal peaks
- Spring break adds a smaller but notable bump
- Recovery trend reflects post-pandemic growth

## Limitations

1. Synthetic — real passenger data depends on pricing, competition, economy
2. No fare/pricing features
3. No route-level data (aggregate network only)
4. No competitor capacity data
5. No fuel price or economic indicators

## How to Improve This Project

1. Add fare class revenue data for revenue management
2. Use route-level forecasts for network planning
3. Include economic indicators (GDP, unemployment)
4. Add competitor schedule data
5. Model disruptions (weather, strikes) separately

## Production Considerations

- Daily passenger forecasting by route and cabin class
- Integration with revenue management systems
- Crew scheduling optimization
- Fuel hedging informed by demand forecasts

## Common Mistakes

1. Not separating business vs leisure travel patterns
2. Ignoring holiday calendar effects
3. Using aggregate forecasts for route-level decisions
4. Not accounting for capacity constraints (you can't exceed aircraft seats)

## Mini Challenge / Exercises

1. Model business routes (weekday peak) vs leisure routes (weekend peak) separately
2. Add a synthetic fare feature and measure demand elasticity
3. Detect disruption events in the residuals
4. Build a monthly passenger forecast for fleet planning

## Final Summary / Key Takeaways

1. Airline passenger demand has strong weekly and seasonal patterns
2. Business travel drives weekday peaks; holidays drive seasonal peaks
3. Revenue management depends on accurate demand forecasts
4. Route-level and cabin-class granularity is needed for operations
5. External factors (economy, pricing) are key missing predictors

In [18]:
import json
metrics = {
    'project': 'Airline Passenger Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Airline Passenger Forecasting — Complete ===")

Metrics saved.

=== Airline Passenger Forecasting — Complete ===
